# Lab 02: Chat Sessions and Streaming (Gemini 3.1)

In this lab, we will learn how to maintain stateful conversations using **Chat Sessions**, how to manually manipulate the conversation history, and how to implement robust **Streaming** with **Gemini 3.1**.

## Objectives
1. Use `client.chats.create` to manage multi-turn conversations.
2. Manually edit and truncate **Chat History**.
3. Implement **Streaming** with interruption handling.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Stateful Chat Sessions

The `chats` object automatically manages the history for you.

In [ ]:
chat = client.chats.create(model="gemini-3.1-flash-lite-preview")

response = chat.send_message("Hi, I'm a scientist working on fusion energy.")
print(f"Gemini: {response.text}")

response = chat.send_message("What is my profession?")
print(f"\nGemini: {response.text}")

## 2. Manual History Manipulation

Sometimes you need to clear old messages to save tokens or inject fake history to guide the model. You can access and modify `chat.history` directly.

In [ ]:
print(f"Current history length: {len(chat.get_history())} messages.")

# 1. Injecting a 'fake' past response from the model
chat.get_history().append(
    types.Content(
        role="model",
        parts=[types.Part.from_text(
            text="We also discussed your recent breakthrough in plasma stabilization."
        )]
    )
)

response = chat.send_message("What was our last discussion about?")
print(f"\nGemini (after injection): {response.text}")

# 2. Truncating history (keeping only the last 2 messages)
chat.history = chat.get_history()[-2:]
print(f"\nHistory truncated to {len(chat.history)} messages.")

## 3. Robust Streaming

Streaming is great for UX, but you must handle cases where the stream might be blocked mid-generation by safety filters or errors.

In [ ]:
prompt = "Write a very long detailed essay about the history of computing."
chat = client.chats.create(model="gemini-3.1-flash-lite-preview")

print("Gemini: ", end="")
try:
    for chunk in chat.send_message_stream(prompt):
        # Check if the chunk has text (it might be empty if blocked)
        if chunk.text:
            print(chunk.text, end="", flush=True)

        # Check if generation was interrupted
        if chunk.candidates[0].finish_reason == "SAFETY":
            print("\n\n[STREAM BLOCKED BY SAFETY FILTER]")
            break
except Exception as e:
    print(f"\n\n[ERROR DURING STREAM]: {e}")

## Summary

In this lab, you learned how to leverage the full power of Chat Sessions:
1. Automating multi-turn state with `client.chats.create`.
2. **Editing and injecting history** to control the conversation context.
3. Implementing **safe and robust streaming** for production environments.